In [13]:
import os
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
from uuid import uuid4
from PIL import Image
load_dotenv()

True

In [4]:
model_id = "openai/clip-vit-base-patch32"

In [6]:
from pathlib import Path
BASE_DIR = Path().resolve().parent
image_root = BASE_DIR/"images"
MODEL_ID = "immich-app/ViT-B-32__laion2b-s34b-b79k"
print(image_root)

F:\Agentic AI\Image-Semantic-Search\images


In [7]:
from langchain_experimental.open_clip import OpenCLIPEmbeddings
embedder = OpenCLIPEmbeddings(
    model_name="ViT-B-32" , 
    checkpoint= "laion2b_s34b_b79k",
    device = "cpu"
    )

f:\Agentic AI\Image-Semantic-Search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
str(image_root/"animals"/"cat.jpeg")

'F:\\Agentic AI\\Image-Semantic-Search\\images\\animals\\cat.jpeg'

In [21]:
img_embed = embedder.embed_image([Path(str(image_root/"animal"/"cat.jpeg"))])

In [24]:
len(img_embed[0])

512

## Data Ingestion

In [14]:
qdrant_api = os.getenv("QDRANT_API_KEY")
qdrant_endpoint = os.getenv("API_ENDPOINT")

In [15]:
from qdrant_client import QdrantClient
from qdrant_client.http import models 
client = QdrantClient(url=qdrant_endpoint , api_key= qdrant_api)

In [33]:
collections = client.get_collections().collections
collections

[]

In [18]:
COLLECTION_NAME = "semantic_image_search"
VECTOR_SIZE = 512

In [36]:
collections = client.get_collections().collections
existing_names = {c.name for c in collections}
existing_names

{'semantic_image_search'}

In [37]:
if COLLECTION_NAME not in existing_names:
    print(f"Creating Collection : {COLLECTION_NAME}")
    client.create_collection(collection_name = COLLECTION_NAME  , vectors_config= models.VectorParams(size=VECTOR_SIZE , distance= models.Distance.COSINE))
else:
    print(f"Collection Already Exist : {COLLECTION_NAME}")

Collection Already Exist : semantic_image_search


In [40]:
def index_image(image_path , category=None):
    image_embed = embedder.embed_image([str(image_path)])[0]
    emb = np.array(image_embed).tolist()
    payload = {"filename" : os.path.basename(image_path) , "path" : image_path , "category" : category}
    client.upsert(
        collection_name= COLLECTION_NAME , 
        points = [
            models.PointStruct(
                id = str(uuid4()) , 
                vector= emb , 
                payload= payload)])
    print(f"Indexed : {image_path}")

In [ ]:
cat_image_path = image_root/"animal"/"cat.jpeg"
# index_image(cat_image_path , category="animal")

In [ ]:
def index_folder(root_folder):
    exts = (".jpg", ".jpeg", ".png", ".webp")
    for dirpath, _, files in os.walk(root_folder):
        category = os.path.basename(dirpath)
        for f in files:
            if f.lower().endswith(exts):
                img_path = os.path.join(dirpath, f)
                # print(img_path,category)
                index_image(img_path,category=category)

In [43]:
index_folder(image_root)

Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\cat.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\crocodile.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\crocodile_1.png
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\dog.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\elephant.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\giraffe.webp
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\horse.webp
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\lion.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\panda.jpg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\tiger.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\zebra.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\flower\lavender.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\flower\lily.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\flower\lotus.j

## Data Retrieval Text 2 Image

In [16]:
def search_text(query,k=5):
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query,
        limit=k,
        with_payload=True,
        with_vectors=True
    )
    return results

In [21]:
query = "image of a cat with angery face"
results = search_text(embedder.embed_query(query),k=3)

In [22]:
for point in results.points:
    print(point.payload, "score =", point.score)

{'filename': 'cat.jpeg', 'path': 'F:\\Agentic AI\\Image-Semantic-Search\\images\\animal\\cat.jpeg', 'category': 'animal'} score = 0.31338164
{'filename': 'cat.jpeg', 'path': 'F:\\Agentic AI\\Image-Semantic-Search\\images\\animal\\cat.jpeg', 'category': 'animal'} score = 0.31338164
{'filename': 'tiger.jpeg', 'path': 'F:\\Agentic AI\\Image-Semantic-Search\\images\\animal\\tiger.jpeg', 'category': 'animal'} score = 0.18903635


In [23]:
for point in results.points:
    print("Payload:", point.payload)
    print("Vector:", point.vector)
    print("Score:", point.score)

Payload: {'filename': 'cat.jpeg', 'path': 'F:\\Agentic AI\\Image-Semantic-Search\\images\\animal\\cat.jpeg', 'category': 'animal'}
Vector: [0.04853688, -0.0014374289, -0.034488074, -0.08087948, 0.026776435, 0.07083485, -0.008278934, -0.048146274, 0.058699567, 0.0140008265, 0.01861771, -0.026211318, -0.024436098, -0.056170173, -0.01332821, 0.025808046, -0.121219955, -0.021169638, -0.010361996, 0.0012653514, 0.015890488, -0.010824875, -0.047347452, -0.011073187, -0.06817598, -0.025547286, -0.054289386, 0.016541831, -0.026753685, -0.016311452, -0.028907659, -0.010116468, -0.050070833, 0.036650874, -0.051859926, -0.0010904277, 0.003514904, 0.08354637, -0.070931435, -0.06484893, 0.06235931, 0.0067744926, 0.065656304, -0.010095247, -0.06366816, -0.056312714, -0.043384288, 0.035559013, 0.011107904, -0.0018562466, -0.005544196, -0.011190538, -0.0047756294, 0.012117726, 0.019831832, -0.0852734, -0.016398985, -0.060443792, -0.02658768, -0.040933724, 0.012819512, 0.052767206, -0.068142556, 0.0353

## Data Retrial Image 2 Image

In [24]:
def search_by_image(image_path,k=5):
    emb = embedder.embed_image([image_path])[0]
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=emb,
        limit=k,
        with_payload=True
    )
    return results


In [27]:
query_image = cat_image_path

In [28]:
results= search_by_image(query_image,k=3)

In [29]:
for point in results.points:
    print(point.payload, "score =", point.score)

{'filename': 'cat.jpeg', 'path': 'F:\\Agentic AI\\Image-Semantic-Search\\images\\animal\\cat.jpeg', 'category': 'animal'} score = 1.0
{'filename': 'cat.jpeg', 'path': 'F:\\Agentic AI\\Image-Semantic-Search\\images\\animal\\cat.jpeg', 'category': 'animal'} score = 1.0
{'filename': 'tiger.jpeg', 'path': 'F:\\Agentic AI\\Image-Semantic-Search\\images\\animal\\tiger.jpeg', 'category': 'animal'} score = 0.6035044


In [30]:
import os
from PIL import Image
from pathlib import Path
import shutil
import uuid

def save_retrieved_images(results, output_dir="retrieved_results"):
    output_dir = Path(output_dir) / uuid.uuid4().hex
    output_dir.mkdir(parents=True, exist_ok=True)

    for idx, point in enumerate(results.points):
        try:
            img_path = point.payload["path"]
            img = Image.open(img_path)

            save_path = output_dir / f"result_{idx}.png"
            img.save(save_path)

        except Exception as e:
            print(f"Error saving image {idx}: {e}")

    print(f"Images saved in → {output_dir}")
    return str(output_dir)


In [31]:
folder = save_retrieved_images(results)

Images saved in → retrieved_results\df7088039c414760bff8db92fe133bde
